# SparkRules — API and Simulation

This notebook shows how to use the SparkRules REST API for rule validation, simulation, and governance.

**Prerequisites:** Start the API server first:
```bash
pip install sparkrules[api]
python -m uvicorn sparkrules.api.app:create_app --factory --port 8042
```

In [ ]:
import httpx
import json

BASE = "http://127.0.0.1:8042"

# Check health
r = httpx.get(f"{BASE}/health")
print("Health:", r.json())

## 1. Validate DRL syntax

In [ ]:
drl = """
rule "credit-check"
  salience 10
  reason_codes ["CR001", "CR002"]
  when
    $app : Application( credit_score < 600 )
  then
    result.decision = "decline";
    result.reason = "low credit score";
end
"""

r = httpx.post(f"{BASE}/rules/validate", json={"drl": drl})
print("Valid:", r.json())

## 2. Simulate rule execution

In [ ]:
r = httpx.post(
    f"{BASE}/simulations",
    json={
        "drl": drl,
        "facts": {"credit_score": 550, "income": 45000},
    },
)
print("Simulation result:")
print(json.dumps(r.json(), indent=2))

In [ ]:
# Fact that does NOT trigger the rule
r2 = httpx.post(
    f"{BASE}/simulations",
    json={
        "drl": drl,
        "facts": {"credit_score": 750, "income": 80000},
    },
)
print("High score result:")
print(json.dumps(r2.json(), indent=2))

## 3. Create a versioned rule

In [ ]:
r = httpx.post(
    f"{BASE}/rules",
    json={
        "rule_handle": "credit-check",
        "group": "underwriting",
        "drl": drl,
    },
)
print("Created rule:")
print(json.dumps(r.json(), indent=2))

## 4. List rule assets

In [ ]:
r = httpx.get(f"{BASE}/rules/assets")
print("Rule assets:")
for asset in r.json():
    print(f"  {asset['rule_handle']} v{asset['version']} (active={asset['is_active']})")

## 5. Counterfactual simulation

Compare what happens with different fact values — useful for "what-if" analysis.

In [ ]:
r = httpx.post(
    f"{BASE}/simulations/counterfactual",
    json={
        "drl": drl,
        "baseline_facts": {"credit_score": 550, "income": 45000},
        "variant_facts": {"credit_score": 650, "income": 45000},
    },
)
print("Counterfactual:")
print(json.dumps(r.json(), indent=2))

## 6. LSP diagnostics

Get editor-style diagnostics for DRL — the same API powers the Workbench Monaco editor.

In [ ]:
r = httpx.post(f"{BASE}/ide/lsp/analyze", json={"drl": drl})
print("LSP diagnostics:", r.json())